## Step 1: Install Dependencies

In [4]:
!pip install ultralytics -q
!pip install scikit-learn seaborn -q
print("✓ All dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.4 MB/s eta 0:00:00
✓ All dependencies installed


## Step 2: Mount Google Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")

Mounted at /content/drive
✓ Google Drive mounted


## Step 3: Import Libraries

In [6]:
import cv2
import os
import json
import pandas as pd
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from datetime import datetime

print("✓ All libraries imported")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ All libraries imported


## Step 4: Configuration

In [7]:
# ============================================
# PATHS
# ============================================

VIDEOS_FOLDER = Path("/content/drive/MyDrive/face_pipeline_project/videos")
OUTPUT_BASE_FOLDER = Path("/content/drive/MyDrive/face_pipeline_project/extracted_frames")
GROUND_TRUTH_FOLDER = Path("/content/drive/MyDrive/face_pipeline_project/ground_truth")
METRICS_FOLDER = Path("/content/drive/MyDrive/face_pipeline_project/metrics")
CHECKPOINT_FILE = Path("/content/drive/MyDrive/face_pipeline_project/checkpoint.json")

# ============================================
# PROCESSING SETTINGS
# ============================================

TARGET_WIDTH = 1280
TARGET_HEIGHT = 720
CONFIDENCE_THRESHOLD = 0.5
FRAME_INTERVAL_SECONDS = 1  # Extract 1 frame every 5 seconds

# Create folders
OUTPUT_BASE_FOLDER.mkdir(exist_ok=True, parents=True)
GROUND_TRUTH_FOLDER.mkdir(exist_ok=True, parents=True)
METRICS_FOLDER.mkdir(exist_ok=True, parents=True)

print(f"Videos folder: {VIDEOS_FOLDER}")
print(f"Output folder: {OUTPUT_BASE_FOLDER}")
print(f"Frame interval: 1 frame every {FRAME_INTERVAL_SECONDS} seconds")

Videos folder: /content/drive/MyDrive/face_pipeline_project/videos
Output folder: /content/drive/MyDrive/face_pipeline_project/extracted_frames
Frame interval: 1 frame every 1 seconds


## Step 5: Checkpoint Functions

In [8]:
def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r') as f:
            checkpoint = json.load(f)
            print(f"✓ Checkpoint loaded: {len(checkpoint.get('completed_videos', []))} videos done")
            return checkpoint
    return {'completed_videos': [], 'current_video': None, 'current_frame': 0, 'total_frames_saved': 0}

def save_checkpoint(checkpoint):
    checkpoint['last_updated'] = datetime.now().isoformat()
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f, indent=2)

def mark_video_complete(checkpoint, video_name, frames_saved):
    checkpoint['completed_videos'].append(video_name)
    checkpoint['current_video'] = None
    checkpoint['current_frame'] = 0
    checkpoint['total_frames_saved'] += frames_saved
    save_checkpoint(checkpoint)

def update_progress(checkpoint, video_name, frame_num):
    checkpoint['current_video'] = video_name
    checkpoint['current_frame'] = frame_num
    if frame_num % 500 == 0:
        save_checkpoint(checkpoint)

checkpoint = load_checkpoint()

✓ Checkpoint loaded: 26 videos done


## Step 6: Verify Videos

In [9]:
video_files = sorted(list(VIDEOS_FOLDER.glob("*.mp4")) + list(VIDEOS_FOLDER.glob("*.MP4")))
completed = set(checkpoint['completed_videos'])
remaining_videos = [v for v in video_files if v.stem not in completed]

print(f"Total videos: {len(video_files)}")
print(f"Completed: {len(completed)}")
print(f"Remaining: {len(remaining_videos)}")

Total videos: 31
Completed: 26
Remaining: 5


## Step 7: Load YOLOv8 Model

In [10]:
print("Loading YOLOv8 model...")
model = YOLO('yolov8n.pt')
print("✓ YOLOv8 model loaded")

Loading YOLOv8 model...
✓ YOLOv8 model loaded


## Step 8: Process Videos (1 frame every 5 seconds)

In [11]:
detections_file = METRICS_FOLDER / "person_detections.csv"
if detections_file.exists():
    all_detections = pd.read_csv(detections_file).to_dict('records')
else:
    all_detections = []

for video_idx, video_path in enumerate(remaining_videos, 1):
    video_name = video_path.stem
    print(f"\n{'='*60}")
    print(f"Processing {video_idx}/{len(remaining_videos)}: {video_path.name}")
    print(f"{'='*60}")

    video_output_folder = OUTPUT_BASE_FOLDER / video_name
    video_output_folder.mkdir(exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"❌ Error opening: {video_path.name}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Calculate frame interval: 1 frame every 5 seconds
    frame_interval = max(1, int(fps * FRAME_INTERVAL_SECONDS))

    print(f"FPS: {fps:.2f}, Total frames: {total_frames}")
    print(f"Frame interval: every {frame_interval} frames ({FRAME_INTERVAL_SECONDS} seconds)")

    # Resume if needed
    start_frame = 0
    saved_count = 0
    if checkpoint['current_video'] == video_name:
        start_frame = checkpoint['current_frame']
        saved_count = len(list(video_output_folder.glob("*.jpg")))
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        print(f"⏩ Resuming from frame {start_frame}")

    frame_count = start_frame
    pbar = tqdm(total=total_frames - start_frame, desc="Processing")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_interval == 0:
            resized_frame = cv2.resize(frame, (TARGET_WIDTH, TARGET_HEIGHT))
            results = model(resized_frame, verbose=False)

            person_count = 0
            person_confidences = []

            for result in results:
                for box in result.boxes:
                    if int(box.cls[0]) == 0 and float(box.conf[0]) >= CONFIDENCE_THRESHOLD:
                        person_count += 1
                        person_confidences.append(float(box.conf[0]))

            all_detections.append({
                'video': video_name,
                'frame_num': frame_count,
                'frame_name': f"frame_{saved_count:06d}.jpg" if person_count > 0 else None,
                'persons_detected': person_count,
                'person_present': 1 if person_count > 0 else 0
            })

            if person_count > 0:
                frame_name = video_output_folder / f"frame_{saved_count:06d}.jpg"
                cv2.imwrite(str(frame_name), resized_frame)
                saved_count += 1

        update_progress(checkpoint, video_name, frame_count)
        frame_count += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    mark_video_complete(checkpoint, video_name, saved_count)
    print(f"✓ Saved {saved_count} frames")

    pd.DataFrame(all_detections).to_csv(detections_file, index=False)

print(f"\n✅ ALL DONE! Total frames: {checkpoint['total_frames_saved']}")


Processing 1/5: 14.54.51-15.17.15[M][0@0][122061]_ch1.mp4
FPS: 25.00, Total frames: 33640
Frame interval: every 25 frames (1 seconds)
⏩ Resuming from frame 13500


Processing: 100%|██████████| 20140/20140 [08:47<00:00, 38.17it/s]


✓ Saved 822 frames

Processing 2/5: 15.11.15-15.36.03[M][0@0][16572]_ch1.mp4
FPS: 25.00, Total frames: 37275
Frame interval: every 25 frames (1 seconds)


Processing: 100%|██████████| 37275/37275 [16:37<00:00, 37.35it/s]


✓ Saved 1142 frames

Processing 3/5: 15.32.02-16.06.52[M][0@0][89109]_ch1.mp4
FPS: 25.00, Total frames: 52309
Frame interval: every 25 frames (1 seconds)


...

✓ Saved 911 frames

Processing 4/5: 15.52.42-16.07.48[M][0@0][17210]_ch1.mp4
FPS: 25.00, Total frames: 22657
Frame interval: every 25 frames (1 seconds)


Processing: 100%|██████████| 22657/22657 [10:02<00:00, 37.59it/s]


✓ Saved 574 frames

Processing 5/5: 16.10.25-16.37.17[M][0@0][123219]_ch1.mp4
FPS: 25.00, Total frames: 40330
Frame interval: every 25 frames (1 seconds)


Processing: 100%|██████████| 40330/40330 [17:57<00:00, 37.44it/s]


✓ Saved 1025 frames

✅ ALL DONE! Total frames: 39139
